In [ ]:
import os
from torch.utils.data import DataLoader

from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger, CSVLogger
import lightning as L

from dataset import DIV2KDataset, TestDataset
from lightning_module import LitESPCN


def get_all_files(path):
    return [path + file for file in os.listdir(path)]

In [11]:
EPOCHS = 50
LR = 1e-3
BATCH_SIZE = 64

In [12]:
train_path = "data/DIV2K_train_HR/"
val_path = "data/DIV2K_valid_HR/"

test_path_5_hr = "data/Set5/GTmod12/"
test_path_5_lr = "data/Set5/LRbicx2/"

test_path_14_hr = "data/Set14/GTmod12/"
test_path_14_lr = "data/Set14/LRbicx2/"

train_dataset = DIV2KDataset(get_all_files(train_path), 2, 17, False)
val_dataset = DIV2KDataset(get_all_files(val_path), 2, 17, False, True, False)

test_dataset = TestDataset(get_all_files(test_path_5_hr) + get_all_files(test_path_14_hr), 
                            get_all_files(test_path_5_lr) + get_all_files(test_path_14_lr),
                            False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE)
val_loader = DataLoader(val_dataset)
test_loader = DataLoader(test_dataset)

### Wo Quantization

In [ ]:
model = LitESPCN(
    scale=2,
    lr=LR,
    weight_decay=0.0,
)

# logger = TensorBoardLogger(
#     save_dir="./tb_logs",
#     name="wo_quant",
# )

logger = CSVLogger(
    save_dir="./tb_logs",
    name="wo_quant",
)

checkpoint_callback = ModelCheckpoint(
    dirpath=os.path.join("./save_model", "wo_quant"),
    filename="espcn-{epoch:03d}-{val_psnr:.2f}",
    monitor="val/psnr",
    mode="max",
    save_top_k=1,
    save_last=True
)

trainer = L.Trainer(
    max_epochs=EPOCHS,
    logger=logger,
    callbacks=[checkpoint_callback],
    devices="auto",
    log_every_n_steps=50,
    deterministic=False,
    default_root_dir=".",
)


trainer.fit(
    model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
)

trainer.test(model, dataloaders=test_loader)

### Unsigned Quant

In [ ]:
model = LitESPCN(
    scale=2,
    quant=True,
    lr=1e-3,
    weight_decay=0.0,
)

logger = TensorBoardLogger(
    save_dir="./tb_logs",
    name="quant_unsigned",
)
# logger = CSVLogger(
#     save_dir="./tb_logs",
#     name="quant_unsigned",
# )

checkpoint_callback = ModelCheckpoint(
    dirpath=os.path.join("./save_model", "quant_unsigned"),
    filename="espcn-{epoch:03d}-{val_psnr:.2f}",
    monitor="val/psnr",
    mode="max",
    save_top_k=1,
    save_last=True
)

trainer = L.Trainer(
    max_epochs=50,
    logger=logger,
    callbacks=[checkpoint_callback],
    devices="auto",
    log_every_n_steps=50,
    deterministic=False,
    default_root_dir=".",
)


trainer.fit(
    model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
    # ckpt_path="save_model/quant_unsigned/last-v4.ckpt"
)

trainer.test(model, dataloaders=test_loader)

### Quant Signed

In [ ]:
model = LitESPCN(
    scale=2,
    quant=True,
    signed=True,
    lr=1e-3,
    weight_decay=0.0,
)

logger = TensorBoardLogger(
    save_dir="./tb_logs",
    name="quant_signed",
)
# logger = CSVLogger(
#     save_dir="./tb_logs",
#     name="quant_unsigned",
# )

checkpoint_callback = ModelCheckpoint(
    dirpath=os.path.join("./save_model", "quant_signed"),
    filename="espcn-{epoch:03d}-{val_psnr:.2f}",
    monitor="val/psnr",
    mode="max",
    save_top_k=1,
    save_last=True
)

trainer = L.Trainer(
    max_epochs=50,
    logger=logger,
    callbacks=[checkpoint_callback],
    devices="auto",
    log_every_n_steps=50,
    deterministic=False,
    default_root_dir=".",
)


trainer.fit(
    model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
)

trainer.test(model, dataloaders=test_loader)